# 6 · Índice documental — o que as atas do Copom dizem (e o que não dizem)

**Objetivo desta fase:** transformar as atas do Copom em um **índice verificável
de argumentos** usados para sustentar o patamar de juros — cada trecho legado
é rastreável até a **ata (nº), data e frase** originais.

**Corpus:** 47 atas da **236ª (20/1/2021) à 282ª (3–4/11/2026)** — PDFs públicos
do BCB (`bcb.gov.br/content/copom/atascopom`), texto extraído com `pypdf`
(805 mil caracteres). A **282ª (3–4/11/2026)** foi incorporada ao rerodar
do extrator em 11/11/2026.

**O que este índice NÃO é:**
- **não é** análise semântica automatizada: as expressões são *regexes* que
  localizam frases candidatas; a leitura é humana;
- **não atribui posições individuais**: as atas registram a **decisão e o
  debate do Comitê**, não votos nominais — não há "quem votou o quê" em
  documento público;
- **não extrapola**: onde o texto não fala, o índice fica vazio (ex.: o termo
  "swap" **não aparece em nenhuma** das 47 atas — e isso é um resultado).

**Perguntas do projeto que este índice documenta:** P1 (os juros se justificam
pela inflação?) e P5 (a política era proposital?) — aqui ficam os *argumentos
escritos* que sustentaram as decisões já quantificadas nos notebooks 00–05.


In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA
from src.processa import indicadores as ind
from src.processa import atas_copom as ac

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
cat = ac.carrega_atas()
print(f"{len(cat)} atas: {cat.num.min()}ª ({cat.data.iloc[0]}) … "
      f"{cat.num.max()}ª ({cat.data.iloc[-1]})")
print(f"texto extraído: {cat.n_caracteres.sum():,} caracteres "
      f"(mín {cat.n_caracteres.min():,} / máx {cat.n_caracteres.max():,})")
cat[["num", "data", "ano", "n_frases"]].head(8)

47 atas: 236ª (2021-01-20) … 282ª (2026-11-04)
texto extraído: 804,649 caracteres (mín 10,744 / máx 24,204)


,num,data,ano,n_frases
0,236,2021-01-20,2021,71
1,237,2021-03-17,2021,70
2,238,2021-05-05,2021,54
3,239,2021-06-16,2021,66
4,240,2021-08-04,2021,60
5,241,2021-09-22,2021,58
6,242,2021-10-27,2021,62
7,243,2021-12-08,2021,60


In [3]:
mat = ac.temas()
temas_ord = [t for t in ac.TEMAS if t in mat.columns]
matriz = mat.groupby("ano")[temas_ord].sum()

fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(matriz.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(temas_ord)), [t.replace(" ", "\n") for t in temas_ord],
              fontsize=9)
ax.set_yticks(range(len(matriz.index)), matriz.index)
for i in range(len(matriz.index)):
    for j in range(len(temas_ord)):
        v = matriz.values[i, j]
        if v:
            ax.text(j, i, int(v), ha="center", va="center", fontsize=9,
                    color="black" if v < matriz.values.max() * 0.6 else "white")
ax.set_title("Atas do Copom 2021–2026: nº de frases por ano e tema (inventário)")
fig.colorbar(im, ax=ax, shrink=0.8, label="frases")
salvar(fig, "06_temas_por_ano")

fig -> viz/figs/06_temas_por_ano.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/06_temas_por_ano.png')

In [4]:
d = ac.decisoes()
d["data"] = pd.to_datetime(d.data)

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.step(d.data, d.selic_depois, where="post", color=AZUL, lw=2,
        label="Selic meta (pós-reunião)")
corte = d.delta_bp.fillna(0) > 0
queda = d.delta_bp.fillna(0) < 0
ax.scatter(d.loc[corte, "data"], d.loc[corte, "selic_depois"], s=42,
           color=VERMELHO, zorder=3, label=f"alta ({int(corte.sum())} reuniões)")
ax.scatter(d.loc[queda, "data"], d.loc[queda, "selic_depois"], s=42,
           color=VERDE, zorder=3, label=f"baixa ({int(queda.sum())} reuniões)")
for _, r in d[corte].iterrows():
    ax.annotate(f"+{int(r.delta_bp)}", (r.data, r.selic_depois),
                textcoords="offset points", xytext=(0, 7), ha="center",
                fontsize=7.5, color=VERMELHO)
ax.set_ylabel("% a.a.")
ax.set_title("Atas 236–281 (2021–2026): decisão a decisão, com o delta em pontos-base")
ax.legend(frameon=False, fontsize=9)
salvar(fig, "06_decisoes_selic")

fig -> viz/figs/06_decisoes_selic.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/06_decisoes_selic.png')

In [5]:
ac.salvar_trechos()
df_trechos = pd.read_parquet(p / "dados" / "processado" / "copom_trechos.parquet")
print(f"trechos indexados: {len(df_trechos):,} em "
      f"{df_trechos.num.nunique()} atas, {df_trechos.tema.nunique()} temas")
df_trechos.tema.value_counts().to_frame("n")

trechos indexados: 1,103 em 47 atas, 10 temas


,n
tema,
risco externo,227
câmbio,214
meta de inflação,208
mercado financeiro,187
risco fiscal,117
credibilidade,109
histórico de desencontro,19
surpresa inflacionária,12
juro real,9


In [6]:
# trechos citáveis dos momentos de virada — cada um rastreável à ata
momentos = [
    (249, "21/9/2022 — pico do aperto (Selic 13,75%)"),
    (252, " 1/2/2023 — última alta de 150 p.p. em ciclo"),
    (267, "11/12/2024 — retomada de alta (+100 p.b.)"),
    (271, "18/6/2025 — topo em 15,00%"),
]
for num, rotulo in momentos:
    sub = df_trechos[(df_trechos.num == num)
                     & df_trechos.tema.isin(["credibilidade", "risco fiscal",
                                             "meta de inflação"])]
    if sub.empty:
        continue
    print("=" * 100)
    print(f"Ata {num} — {rotulo}  [tema: {sub.tema.iloc[0]}]")
    print(sub.trecho.iloc[0][:700])

Ata 249 — 21/9/2022 — pico do aperto (Selic 13,75%)  [tema: credibilidade]
A lguns membros enfatizaram que houve um aumento nas expectativas de prazos mais longos em grande parte dos países emergentes, refletindo a magnitude e a persistência dos choques globais ainda em curso. Por fim, todos os membros concordam que é papel primordial e fundamental da autoridade monetária a condução de uma política monetária compatível com a ancoragem das expectativas em prazos mais longos, fortalecendo continuamente a sua credibilidade e reduzindo o custo desinflacionário futuro. O Comitê ressalta que, em seus cenários para a inflação, permanecem fatores de risco em ambas as direções.
Ata 252 —  1/2/2023 — última alta de 150 p.p. em ciclo  [tema: meta de inflação]
br Ata da Reunião do Comitê de Política Monetária — Copom 31 de janeiro e 1º de fevereiro de 2023 252ª
252ª Reunião COPOM 31 de janeiro e 1º de fevereiro 2 bcb. br Os membros do Copom analisaram a evolução recente e as perspectivas para a ec

## O que o índice mostra

1. **A ata é coletiva.** O documento registra a decisão e o debate do Comitê
   ("todos os membros concordam…"), **sem votos nominais** — a pergunta
   "quem defendeu o patamar" **não tem resposta em documento público do Copom**;
   nomes individuais exigiriam fonte adicional (discursos, votos em CPI,
   entrevistas), que fica para o corpus complementar.

2. **Os argumentos escritos mudam de regime**, coerente com os dados dos
   notebooks 00–05:
   - **2021–2022 (aperto de 2,00% → 13,75%):** dominam *câmbio*, *mercado
     financeiro* e *surpresa inflacionária* — a ata reage a choques e
     desancoragem ("sequência de surpresas", "acima do esperado");
   - **2023–2024 (corte e nova alta):** *risco fiscal* sobe de 15 → 26
     frases/ano e *credibilidade* de 1 → 25 — a ata passa a sustentar o
     patamar alto por **ancoragem e sustentabilidade fiscal**;
   - **2025 (topo 15%):** *credibilidade* é o tema mais frequente (44) —
     "a desancoragem das expectativas é um fator de desconforto comum a todos
     os membros" (ata 271, 18/6/2025).

3. **Silêncios também documentam.** "Swap" = 0 ocorrências; "juro real ex ante"
   = 1 em 47 atas. O Comitê **não** justifica o patamar explicitamente em
   termos de juro real ex ante — o prêmio de ~6 p.p. sobre a regra de Taylor
   (notebook 05) é **inferido dos dados**, não citado nas atas.

**Reprodutibilidade:** `PYTHONPATH=. python -m src.processa.atas_copom`
regenera a matriz e o parquet `dados/processado/copom_trechos.parquet`
(1.103 trechos indexados, incluindo a ata 282).

**Limitações:** regex ≠ semântica (falsos positivos/negativos possíveis);
trechos precisam de leitura humana antes de citação; o corpus complementar dos
**discursos** está no notebook 07; **manifestos de entidades** seguem
não coletados.